# imports + setup

In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torchvision.models as models
from medmnist import BloodMNIST
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_folder   = "/blue/bme6938/gael.garcia/bme6938-project-2/data"
models_folder = "/blue/bme6938/gael.garcia/bme6938-project-2/models"
cell_types    = ['basophil', 'eosinophil', 'erythroblast', 'ig',
                 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']

# load data + best model (EfficientNet)

In [2]:
test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_dataset = BloodMNIST(split='test', download=False, size=64, root=data_folder, transform=test_transforms)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 8)
model.load_state_dict(torch.load(os.path.join(models_folder, 'best_efficientnet.pth'), map_location=device))
model = model.to(device)
model.eval()
print("model loaded!")

FileNotFoundError: [Errno 2] No such file or directory: '/blue/bme6938/gael.garcia/bme6938-project-2/models/best_efficientnet.pth'

# sample predictions

In [3]:
sample_imgs, sample_labels = next(iter(test_loader))
with torch.no_grad():
    _, preds = model(sample_imgs[:8].to(device)).max(1)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = sample_imgs[i].permute(1, 2, 0).numpy()
    img = np.clip(img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406], 0, 1)
    ax.imshow(img)
    true = cell_types[sample_labels[i].item()]
    pred = cell_types[preds[i].item()]
    ax.set_title(f'true: {true}\npred: {pred}', color='green' if true == pred else 'red', fontsize=9)
    ax.axis('off')

plt.suptitle('EfficientNet-B0 — sample predictions (green=correct, red=incorrect)')
plt.tight_layout()
plt.show()

RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same